In [3]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import ruptures as rpt
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# Pairwise GGS-like Change Point Detection
# ------------------------------------------------------------
# Input  : ./tvpvar_input_scaled.csv
# Output : ./result_pair/
#
# 목적:
# - SOLVPN-X pairwise regime detection
# - noisy segmentation 억제
# - 짧은 regime 강한 penalty 적용
#
# 핵심 수정:
# 1. min_size 증가
# 2. penalty 증가
# 3. cp 개수 감소 유도
# 4. 짧은 segment 추가 penalty 적용
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")

RESULT_DIR = BASE_DIR / "result_pair"

PLOT_DIR = RESULT_DIR / "plots"

RESULT_DIR.mkdir(parents=True, exist_ok=True)

PLOT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"

DATE_COL = "Date"

TARGET = "dlog_SOLVPN"

OTHERS = [
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "dlog_OIL",
    "dlog_GAS",
    "d_UST10Y",
    "d_VIX",
    "d_PJM_LMP",
]

# ============================================================
# 핵심 tuning
# ============================================================

# Pairwise는 noisy하므로 robust regime만 남기기
TARGET_MIN_CP = 1
TARGET_MAX_CP = 4

# l2 제거
MODEL_LIST = ["rbf", "normal"]

# 더 강한 smoothing
PENALTY_LIST = [10, 15, 20, 30, 50]

# 최소 regime 길이 증가
MIN_SIZE_LIST = [40, 60, 80, 120]

# 짧은 segment penalty 기준
SHORT_SEGMENT_THRESHOLD = 40

# 짧은 segment 발생 시 추가 penalty
SHORT_SEGMENT_PENALTY = 50

# ============================================================
# Output Files
# ============================================================

OUT_GRID = RESULT_DIR / "pair_ggs_grid_search.csv"

OUT_CP = RESULT_DIR / "pair_ggs_change_points.csv"

OUT_SEGMENTS = RESULT_DIR / "pair_ggs_segments.csv"

OUT_STATS = RESULT_DIR / "pair_ggs_segment_stats.csv"

OUT_SUMMARY = RESULT_DIR / "pair_ggs_change_points_summary.csv"

# ============================================================
# Helpers
# ============================================================

def load_data():

    df = pd.read_csv(INPUT_FILE)

    if DATE_COL not in df.columns:
        raise ValueError(f"{DATE_COL} not found.")

    cols = [TARGET] + OTHERS

    missing = [c for c in cols if c not in df.columns]

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df[DATE_COL] = pd.to_datetime(df[DATE_COL])

    df = (
        df[[DATE_COL] + cols]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .sort_values(DATE_COL)
        .reset_index(drop=True)
    )

    return df


def run_segmentation(
    X,
    model_name,
    penalty,
    min_size,
):

    try:

        model = rpt.Pelt(
            model=model_name,
            min_size=min_size,
        ).fit(X)

        cps = model.predict(pen=penalty)

        cps = [
            int(cp)
            for cp in cps
            if cp < len(X)
        ]

        return cps, ""

    except Exception as e:

        return [], repr(e)


def get_segment_lengths(change_idx, n_obs):

    bounds = [0] + change_idx + [n_obs]

    lengths = []

    for i in range(len(bounds) - 1):

        lengths.append(
            bounds[i + 1] - bounds[i]
        )

    return lengths


# ============================================================
# 핵심 scoring 함수
# ============================================================

def score_candidate(
    n_cp,
    segment_lengths,
):

    score = 0

    # --------------------------------------------------------
    # 1. cp 개수 penalty
    # --------------------------------------------------------
    if TARGET_MIN_CP <= n_cp <= TARGET_MAX_CP:

        score += 0

    elif n_cp == 0:

        score += 100

    elif n_cp < TARGET_MIN_CP:

        score += 20

    else:

        score += (n_cp - TARGET_MAX_CP) * 15

    # --------------------------------------------------------
    # 2. 짧은 segment penalty
    # --------------------------------------------------------
    min_seg = min(segment_lengths)

    if min_seg < SHORT_SEGMENT_THRESHOLD:

        score += SHORT_SEGMENT_PENALTY

    # --------------------------------------------------------
    # 3. 너무 짧은 segment 개수 penalty
    # --------------------------------------------------------
    n_short = sum([
        x < SHORT_SEGMENT_THRESHOLD
        for x in segment_lengths
    ])

    score += n_short * 10

    # --------------------------------------------------------
    # 4. segment 길이 분산 penalty
    # --------------------------------------------------------
    seg_std = np.std(segment_lengths)

    score += seg_std * 0.05

    return score


def choose_best_candidate(grid_df):

    valid = grid_df[
        grid_df["error_msg"] == ""
    ].copy()

    if valid.empty:
        raise RuntimeError("No valid candidates.")

    valid = valid.sort_values(
        "score",
        ascending=True,
    ).reset_index(drop=True)

    return valid.iloc[0]


def make_segment_table(
    pair_name,
    change_idx,
    dates,
):

    bounds = [0] + change_idx + [len(dates)]

    rows = []

    for seg_id in range(len(bounds) - 1):

        start_idx = bounds[seg_id]

        end_exclusive = bounds[seg_id + 1]

        end_idx = end_exclusive - 1

        rows.append({
            "pair": pair_name,
            "segment_id": seg_id,
            "start_idx": start_idx,
            "end_idx": end_idx,
            "start_date": dates.iloc[start_idx],
            "end_date": dates.iloc[end_idx],
            "length": end_exclusive - start_idx,
        })

    return pd.DataFrame(rows)


def make_segment_stats(
    pair_name,
    pair_cols,
    data,
    segments_df,
):

    rows = []

    for _, seg in segments_df.iterrows():

        seg_id = int(seg["segment_id"])

        start_idx = int(seg["start_idx"])

        end_idx = int(seg["end_idx"])

        seg_data = data.iloc[start_idx:end_idx + 1]

        for feature in pair_cols:

            values = seg_data[feature].values

            rows.append({
                "pair": pair_name,
                "segment_id": seg_id,
                "feature": feature,
                "mean": float(np.mean(values)),
                "std": float(np.std(values, ddof=0)),
                "var": float(np.var(values, ddof=0)),
                "min": float(np.min(values)),
                "max": float(np.max(values)),
            })

    return pd.DataFrame(rows)


def plot_pair(
    pair_name,
    pair_cols,
    data,
    change_idx,
):

    dates = data[DATE_COL]

    plt.figure(figsize=(14, 6))

    for col in pair_cols:

        plt.plot(
            dates,
            data[col],
            linewidth=1.0,
            alpha=0.85,
            label=col,
        )

    for cp in change_idx:

        plt.axvline(
            dates.iloc[cp],
            linestyle="--",
            linewidth=1.2,
            alpha=0.8,
        )

    plt.title(f"Pairwise GGS - {pair_name}")

    plt.xlabel("Date")

    plt.ylabel("Standardized value")

    plt.legend()

    plt.tight_layout()

    out_path = (
        PLOT_DIR /
        f"{pair_name}.png"
    )

    plt.savefig(out_path, dpi=300)

    plt.close()


# ============================================================
# Main
# ============================================================

def main():

    print("======================================")
    print("Pairwise GGS-like Detection")
    print("======================================")

    data = load_data()

    all_grid_rows = []

    all_cp_rows = []

    all_segment_rows = []

    all_stats_rows = []

    summary_rows = []

    # =====================================================
    # Pair Loop
    # =====================================================
    for other in OTHERS:

        pair_cols = [TARGET, other]

        pair_name = f"{TARGET}__{other}"

        print()
        print("======================================")
        print(f"PAIR : {pair_name}")
        print("======================================")

        pair_df = data[
            [DATE_COL] + pair_cols
        ].copy()

        X = pair_df[pair_cols].values

        dates = pair_df[DATE_COL]

        # -------------------------------------------------
        # Grid Search
        # -------------------------------------------------
        grid_rows = []

        for model_name in MODEL_LIST:

            for min_size in MIN_SIZE_LIST:

                for penalty in PENALTY_LIST:

                    cps, error_msg = run_segmentation(
                        X=X,
                        model_name=model_name,
                        penalty=penalty,
                        min_size=min_size,
                    )

                    segment_lengths = get_segment_lengths(
                        cps,
                        len(X),
                    )

                    score = score_candidate(
                        n_cp=len(cps),
                        segment_lengths=segment_lengths,
                    )

                    grid_rows.append({
                        "pair": pair_name,
                        "model": model_name,
                        "min_size": min_size,
                        "penalty": penalty,
                        "n_cp": len(cps),
                        "min_segment_length": min(segment_lengths),
                        "max_segment_length": max(segment_lengths),
                        "segment_lengths": "|".join(
                            map(str, segment_lengths)
                        ),
                        "score": score,
                        "change_idx": "|".join(map(str, cps)),
                        "change_dates": "|".join(
                            dates.iloc[cps].astype(str).tolist()
                        ) if len(cps) > 0 else "",
                        "error_msg": error_msg,
                    })

        grid_df = pd.DataFrame(grid_rows)

        best = choose_best_candidate(grid_df)

        best_model = best["model"]

        best_min_size = int(best["min_size"])

        best_penalty = float(best["penalty"])

        # -------------------------------------------------
        # Final Run
        # -------------------------------------------------
        change_idx, error_msg = run_segmentation(
            X=X,
            model_name=best_model,
            penalty=best_penalty,
            min_size=best_min_size,
        )

        if error_msg:

            print("[ERROR]", error_msg)

            continue

        print("model      :", best_model)

        print("min_size   :", best_min_size)

        print("penalty    :", best_penalty)

        print("n_cp       :", len(change_idx))

        print("cp idx     :", change_idx)

        # -------------------------------------------------
        # Change Point Table
        # -------------------------------------------------
        cp_df = pd.DataFrame({
            "pair": pair_name,
            "cp_id": np.arange(
                1,
                len(change_idx) + 1
            ),
            "change_idx": change_idx,
            "change_date": dates.iloc[
                change_idx
            ].values,
        })

        # -------------------------------------------------
        # Segment Table
        # -------------------------------------------------
        segments_df = make_segment_table(
            pair_name,
            change_idx,
            dates,
        )

        # -------------------------------------------------
        # Segment Stats
        # -------------------------------------------------
        stats_df = make_segment_stats(
            pair_name,
            pair_cols,
            pair_df,
            segments_df,
        )

        # -------------------------------------------------
        # Plot
        # -------------------------------------------------
        plot_pair(
            pair_name,
            pair_cols,
            pair_df,
            change_idx,
        )

        # -------------------------------------------------
        # Summary
        # -------------------------------------------------
        summary_rows.append({
            "pair": pair_name,
            "model": best_model,
            "min_size": best_min_size,
            "penalty": best_penalty,
            "n_cp": len(change_idx),
        })

        # -------------------------------------------------
        # Collect
        # -------------------------------------------------
        all_grid_rows.append(grid_df)

        all_cp_rows.append(cp_df)

        all_segment_rows.append(segments_df)

        all_stats_rows.append(stats_df)

    # =====================================================
    # Save
    # =====================================================
    final_grid = pd.concat(
        all_grid_rows,
        ignore_index=True,
    )

    final_cp = pd.concat(
        all_cp_rows,
        ignore_index=True,
    )

    final_segments = pd.concat(
        all_segment_rows,
        ignore_index=True,
    )

    final_stats = pd.concat(
        all_stats_rows,
        ignore_index=True,
    )

    final_summary = pd.DataFrame(summary_rows)

    final_grid.to_csv(
        OUT_GRID,
        index=False,
        encoding="utf-8-sig",
    )

    final_cp.to_csv(
        OUT_CP,
        index=False,
        encoding="utf-8-sig",
    )

    final_segments.to_csv(
        OUT_SEGMENTS,
        index=False,
        encoding="utf-8-sig",
    )

    final_stats.to_csv(
        OUT_STATS,
        index=False,
        encoding="utf-8-sig",
    )

    final_summary.to_csv(
        OUT_SUMMARY,
        index=False,
        encoding="utf-8-sig",
    )

    print()
    print("======================================")
    print("Saved Files")
    print("======================================")

    print(OUT_GRID)
    print(OUT_CP)
    print(OUT_SEGMENTS)
    print(OUT_STATS)
    print(OUT_SUMMARY)
    print(PLOT_DIR)

    print()
    print("Finished.")


if __name__ == "__main__":
    main()

Pairwise GGS-like Detection

PAIR : dlog_SOLVPN__dlog_COPPER
model      : normal
min_size   : 120
penalty    : 10.0
n_cp       : 3
cp idx     : [135, 260, 470]

PAIR : dlog_SOLVPN__dlog_GOLD
model      : normal
min_size   : 80
penalty    : 15.0
n_cp       : 3
cp idx     : [85, 170, 455]

PAIR : dlog_SOLVPN__dlog_SILVER
model      : normal
min_size   : 120
penalty    : 15.0
n_cp       : 1
cp idx     : [285]

PAIR : dlog_SOLVPN__dlog_DXY
model      : normal
min_size   : 120
penalty    : 15.0
n_cp       : 2
cp idx     : [215, 380]

PAIR : dlog_SOLVPN__dlog_OIL
model      : normal
min_size   : 120
penalty    : 10.0
n_cp       : 3
cp idx     : [205, 325, 470]

PAIR : dlog_SOLVPN__dlog_GAS
model      : normal
min_size   : 120
penalty    : 15.0
n_cp       : 2
cp idx     : [220, 435]

PAIR : dlog_SOLVPN__d_UST10Y
model      : normal
min_size   : 120
penalty    : 15.0
n_cp       : 3
cp idx     : [120, 320, 485]

PAIR : dlog_SOLVPN__d_VIX
model      : normal
min_size   : 120
penalty    : 50.0
n_